## Setup

In [1]:
from dotenv import load_dotenv
import os, anthropic

load_dotenv(override=True)   

key = os.getenv("ANTHROPIC_API_KEY")
print(repr(key[:10]), len(key), repr(key[-4:]))

client = anthropic.Anthropic()
print(repr(client.api_key[:10]), len(client.api_key))

'sk-ant-usr' 106 'nAAA'
'sk-ant-usr' 106


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
import anthropic
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import os
print(os.getenv("ANTHROPIC_API_KEY") is not None)

True


In [5]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Dustin\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

## Import and Processing

In [6]:
df = pd.read_csv("IMDB-Dataset.csv")
df.tail(2)

,review,sentiment
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [7]:
imdb_df = df.copy()

In [8]:
imdb_df.shape

(50000, 2)

In [9]:
imdb_df = imdb_df.sample(300, random_state=1).reset_index(drop=True)
imdb_df.shape

(300, 2)

In [10]:
imdb_df = imdb_df[['review', 'sentiment']]
imdb_df.head()

,review,sentiment
0,With No Dead Heroes you get stupid lines like ...,negative
1,I thought maybe... maybe this could be good. A...,negative
2,An elite American military team which of cours...,negative
3,Ridiculous horror film about a wealthy man (Jo...,negative
4,"Well, if you are one of those Katana's film-nu...",positive


## Claude API (Haiku)

In [11]:
system_prompt = '''You classify the sentiment of reviews written.
Reply with exactly one word: positive, neutral, or negative.
Currently, the dataset only has positive and negative but update this if possible.
Sarcasm counts as its true sentiment (for example, "I laughed all the way through this rotten movie." is negative).'''

In [12]:
VALID_LABELS = ['positive', 'neutral', 'negative']

In [13]:
def compute_claude_sentiment(text):
    response = client.messages.create(
        model='claude-haiku-5-5',
        max_tokens=10,
        thinking={"type": "disabled"},
        system=system_prompt,
        messages=[{'role': 'user', 'content': f"Review: {text}"}]
    )
    # Grab the first text block, whatever position it's in
    answer = next((b.text for b in response.content if b.type == "text"), "")
    answer = answer.strip().strip(".").lower()
    return answer if answer in VALID_LABELS else 'neutral'

In [14]:
compute_claude_sentiment('This movie was so frustrating')

'negative'

## Applying Claude to Data

In [15]:
imdb_df['claude_sentiment'] = imdb_df['review'].apply(compute_claude_sentiment)

imdb_df[['review', 'sentiment', 'claude_sentiment']].head()

,review,sentiment,claude_sentiment
0,With No Dead Heroes you get stupid lines like ...,negative,negative
1,I thought maybe... maybe this could be good. A...,negative,negative
2,An elite American military team which of cours...,negative,negative
3,Ridiculous horror film about a wealthy man (Jo...,negative,negative
4,"Well, if you are one of those Katana's film-nu...",positive,positive


In [16]:
imdb_df

,review,sentiment,claude_sentiment
0,With No Dead Heroes you get stupid lines like ...,negative,negative
1,I thought maybe... maybe this could be good. A...,negative,negative
2,An elite American military team which of cours...,negative,negative
3,Ridiculous horror film about a wealthy man (Jo...,negative,negative
4,"Well, if you are one of those Katana's film-nu...",positive,positive
...,...,...,...
295,Guy Pearce looks like and acts like a Calvin K...,negative,negative
296,"In an otherwise good review, loleralacartelort...",negative,positive
297,"While the original titillates the intellect, t...",negative,negative
298,"I am a Jane Eyre lover and a purist, and this ...",positive,positive


## Vader Sentiment (Comparison)

In [17]:
sia = SentimentIntensityAnalyzer()

def compute_vader_sentiment(text):
    score = sia.polarity_scores(text)['compound']

    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    return 'neutral'

In [18]:
imdb_df['vader_sentiment'] = imdb_df['review'].apply(compute_vader_sentiment)

In [19]:
imdb_df

,review,sentiment,claude_sentiment,vader_sentiment
0,With No Dead Heroes you get stupid lines like ...,negative,negative,negative
1,I thought maybe... maybe this could be good. A...,negative,negative,positive
2,An elite American military team which of cours...,negative,negative,positive
3,Ridiculous horror film about a wealthy man (Jo...,negative,negative,negative
4,"Well, if you are one of those Katana's film-nu...",positive,positive,positive
...,...,...,...,...
295,Guy Pearce looks like and acts like a Calvin K...,negative,negative,positive
296,"In an otherwise good review, loleralacartelort...",negative,positive,negative
297,"While the original titillates the intellect, t...",negative,negative,positive
298,"I am a Jane Eyre lover and a purist, and this ...",positive,positive,positive


## Comparison of Models

In [20]:
pd.crosstab(imdb_df['sentiment'], imdb_df['claude_sentiment'])

claude_sentiment,negative,neutral,positive
sentiment,,,
negative,141,7,4
positive,2,8,138


In [21]:
pd.crosstab(imdb_df['sentiment'], imdb_df['vader_sentiment'])

vader_sentiment,negative,neutral,positive
sentiment,,,
negative,89,2,61
positive,20,0,128


In [22]:
imdb_df.to_csv('IMDB_reviews_with_claude_sentiment.csv', index=False)